In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer

df = pd.read_csv("titanic_data_updated.csv")

In [2]:
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
284,285,no,first,"Smith, Mr. Richard William",male,NaN,0,0,113056,26.0000,A19,S
605,606,no,third,"Lindell, Mr. Edvard Bengtsson",male,36.0,1,0,349910,15.5500,NaN,S
405,406,no,second,"Gale, Mr. Shadrach",male,34.0,1,0,28664,21.0000,NaN,S
547,548,yes,second,"Padro y Manent, Mr. Julian",male,NaN,0,0,SC/PARIS 2146,13.8625,NaN,C
270,271,no,first,"Cairns, Mr. Alexander",male,NaN,0,0,113798,31.0000,NaN,S


Deleting useless columns

In [3]:
df.drop(['PassengerId','Name','Ticket'],axis=1,inplace=True)

Combining 'SibSp' and 'Parch' columns

In [4]:
df['FamilyMembers'] = df['SibSp'] + df['Parch'] + 1 # 1 is for the person him/herself

In [5]:
df.drop(columns=['SibSp','Parch'],inplace=True)

In [6]:
df.sample(2)

,Survived,Pclass,Sex,Age,Fare,Cabin,Embarked,FamilyMembers
451,no,third,male,NaN,19.9667,NaN,S,2
407,yes,second,male,3.0,18.7500,NaN,S,3


Train Test Split

In [7]:
x = df.drop(columns='Survived') # features
y = df['Survived'] # target

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=42)

In [8]:
x_test.sample(2)

,Pclass,Sex,Age,Fare,Cabin,Embarked,FamilyMembers
218,first,female,32.0,76.2917,D15,C,1
840,third,male,20.0,7.9250,NaN,S,1


Imputation

In [9]:
imputer_transformer = ColumnTransformer(
    transformers=[
        ('age',SimpleImputer(missing_values=np.nan,strategy='mean'),['Age']),
        ('embarked',SimpleImputer(missing_values=np.nan,strategy='most_frequent'),['Embarked']),
        ('cabin',SimpleImputer(missing_values=np.nan,strategy='constant',fill_value='Missing',add_indicator=True),['Cabin'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

imputer_transformer.set_output(transform='pandas')

# fit
imputer_transformer.fit(x_train)

# transform
x_train = imputer_transformer.transform(x_train)
x_test = imputer_transformer.transform(x_test)

In [10]:
x_train.isnull().sum()

Age                       0
Embarked                  0
Cabin                     0
missingindicator_Cabin    0
Pclass                    0
Sex                       0
Fare                      0
FamilyMembers             0
dtype: int64

In [11]:
x_test.isnull().sum()

Age                       0
Embarked                  0
Cabin                     0
missingindicator_Cabin    0
Pclass                    0
Sex                       0
Fare                      0
FamilyMembers             0
dtype: int64

In [12]:
x_train.sort_values(by='Age').tail(10)

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers
555,62.0,S,Missing,True,first,male,26.5500,1
252,62.0,S,C87,False,first,male,26.5500,1
483,63.0,S,Missing,True,third,female,9.5875,1
438,64.0,S,C23 C25 C27,False,first,male,263.0000,6
456,65.0,S,E38,False,first,male,26.5500,1
745,70.0,S,B22,False,first,male,71.0000,3
672,70.0,S,Missing,True,second,male,10.5000,1
116,70.5,Q,Missing,True,third,male,7.7500,1
851,74.0,S,Missing,True,third,male,7.7750,1
630,80.0,S,A23,False,first,male,30.0000,1


### Outliers Handling

Age Outliers

In [13]:
mean_age = x_train['Age'].mean()
std_age = x_train['Age'].std()

x_train['z_score_age'] = (x_train['Age']-mean_age)/std_age

outliers_age = x_train[abs(x_train['z_score_age'])>3]

print(f"Number of outliers: {len(outliers_age)}")
# display(outliers_age)
# x_train.sort_values(by='Age').tail(10)

Number of outliers: 5


Fare outliers

In [14]:
mean_fare = x_train['Fare'].mean()
std_fare = x_train['Fare'].std()

x_train['z_score_fare'] = (x_train['Fare']-mean_fare)/std_fare
outliers_fare = x_train[abs(x_train['z_score_fare'])>3]

print(f"Number of outlier fare: {len(outliers_fare)}")
display(outliers_fare.head())

Number of outlier fare: 17


,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers,z_score_age,z_score_fare
118,24.0,C,B58 B60,False,first,male,247.5208,2,-0.423174,4.135780
716,38.0,C,C45,False,first,female,227.5250,1,0.654222,3.751020
377,27.0,C,C82,False,first,male,211.5000,3,-0.192303,3.442666
742,21.0,C,B57 B59 B63 B66,False,first,female,262.3750,5,-0.654044,4.421605
380,42.0,C,Missing,True,first,female,227.5250,1,0.962049,3.751020


Fare outliers using IQR

In [15]:
fare_q1 = x_train['Fare'].quantile(0.25)
fare_q3 = x_train['Fare'].quantile(0.75)

fare_iqr = fare_q3 - fare_q1

max_range = fare_q3 + (1.5 * fare_iqr)
min_range = max(0,fare_q1 - (1.5 * fare_iqr)) # becasue the value goes to negative side, so capping at 0

print(f"Max: {max_range}\nMin: {min_range}")

fare_outliers = x_train[(x_train['Fare']<min_range) | (x_train['Fare'] > max_range)]
print("Number of fare outliers: ",len(fare_outliers))
display(fare_outliers.sample(5))

Max: 64.3625
Min: 0
Number of fare outliers:  96


,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers,z_score_age,z_score_fare
641,24.000000,C,B35,False,first,female,69.3000,1,-4.231736e-01,0.706447
332,38.000000,S,C91,False,first,male,153.4625,2,6.542216e-01,2.325906
334,29.498846,S,Missing,True,first,female,133.6500,2,2.734055e-16,1.944673
445,4.000000,S,A34,False,first,male,81.8583,3,-1.962310e+00,0.948094
689,15.000000,S,B5,False,first,female,211.3375,2,-1.115785e+00,3.439539


In [16]:
# omitting outliers age from the x_train data
x_train = x_train[abs(x_train['z_score_age']) <=3]
x_train

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers,z_score_age,z_score_fare
331,45.500000,S,C124,False,first,male,28.5000,1,1.231398e+00,-0.078628
733,23.000000,S,Missing,True,second,male,13.0000,1,-5.001304e-01,-0.376880
382,32.000000,S,Missing,True,third,male,7.9250,1,1.924808e-01,-0.474533
704,26.000000,S,Missing,True,third,male,7.8542,2,-2.692600e-01,-0.475896
813,6.000000,S,Missing,True,third,female,31.2750,7,-1.808396e+00,-0.025232
...,...,...,...,...,...,...,...,...,...,...
106,21.000000,S,Missing,True,third,female,7.6500,1,-6.540440e-01,-0.479825
270,29.498846,S,Missing,True,first,male,31.0000,1,2.734055e-16,-0.030523
860,41.000000,S,Missing,True,third,male,14.1083,3,8.850920e-01,-0.355554
435,14.000000,S,B96 B98,False,first,female,120.0000,4,-1.192742e+00,1.682019


In [17]:
# Capping 'Fare' at 0 to max range of IQR outliers
x_train['Fare'] = x_train['Fare'].clip(min_range,max_range)
print(x_train['Fare'].min())
print(x_train['Fare'].max())

0.0
64.3625


In [18]:
# x_train.sort_values(by='Age').tail(5)
x_train.sort_values(by='Fare').tail(5)

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers,z_score_age,z_score_fare
856,45.000000,S,Missing,True,first,female,64.3625,3,1.192919e+00,2.545346
737,35.000000,C,B101,False,first,male,64.3625,1,4.233512e-01,9.231235
180,29.498846,S,Missing,True,third,female,64.3625,11,2.734055e-16,0.711258
792,29.498846,S,Missing,True,third,female,64.3625,11,2.734055e-16,0.711258
700,18.000000,C,C62 C64,False,first,female,64.3625,2,-8.849144e-01,3.751020


In [19]:
x_train = x_train.drop(columns=['z_score_age', 'z_score_fare'], errors='ignore')
x_test = x_test.drop(columns=['z_score_age', 'z_score_fare'], errors='ignore')

In [20]:
x_train.sample(2)

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers
708,22.0,S,Missing,True,first,female,64.3625,1
735,28.5,S,Missing,True,third,male,16.1000,1


### Encoding and Scaling

In [21]:
# making a column 'Cabin_deck' by taking the first character of cabin name
x_train['Cabin_deck'] = x_train['Cabin'].astype(str).str[0];
x_test['Cabin_deck'] = x_test['Cabin'].astype(str).str[0];

In [22]:
x_train.sample(2)

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,Fare,FamilyMembers,Cabin_deck
378,20.0,C,Missing,True,third,male,4.0125,1,M
660,50.0,S,Missing,True,first,male,64.3625,3,M


In [23]:
encoder_scaler = ColumnTransformer(
    transformers= [
        ('pclass',OrdinalEncoder(categories=[['third','second','first']]),['Pclass']),
        ('sex_embarked_cabin',OneHotEncoder(sparse_output=False,drop='first'),['Embarked','Sex','Cabin_deck']),
        ('age',StandardScaler(),['Age']),
        ('fare_family',MinMaxScaler(),['Fare','FamilyMembers'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

encoder_scaler.set_output(transform='pandas')

encoder_scaler.fit(x_train)

x_train = encoder_scaler.transform(x_train)
x_test = encoder_scaler.transform(x_test)

In [24]:
x_train.drop('Cabin',axis=1,inplace=True)
x_test.drop('Cabin',axis=1,inplace=True)

In [25]:
x_train.sample(2)

,Pclass,Embarked_Q,Embarked_S,Sex_male,Cabin_deck_B,Cabin_deck_C,Cabin_deck_D,Cabin_deck_E,Cabin_deck_F,Cabin_deck_G,Cabin_deck_M,Cabin_deck_T,Age,Fare,FamilyMembers,missingindicator_Cabin
551,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.175332,0.403962,0.0,True
685,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.335313,0.646016,0.3,True


In [26]:
x_test.sample(2)

,Pclass,Embarked_Q,Embarked_S,Sex_male,Cabin_deck_B,Cabin_deck_C,Cabin_deck_D,Cabin_deck_E,Cabin_deck_F,Cabin_deck_G,Cabin_deck_M,Cabin_deck_T,Age,Fare,FamilyMembers,missingindicator_Cabin
802,2.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.455182,1.864440,0.3,False
736,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.504471,0.534084,0.4,True
